# ニューラルネットワーク

iris datasetを例に，他クラス分類をMLPでやってみよう！

In [25]:
import numpy as np
import torch
import torch.nn as nn 

# データセット関係
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 可視化関係
import matplotlib.pyplot as plt
import matplotlib_fontja
import seaborn as sns

In [32]:
# Irisデータセットを読み込む
iris = load_iris()
X = iris.data
y = iris.target

# まず訓練データ60%と一時データ40%に層化分割する
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# 一時データ40%をバリデーション10%とテスト30%に層化分割する
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.75, random_state=42, stratify=y_temp
)

# 標準化は訓練データでfitし，バリデーションとテストには同じ変換を適用する
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# NumPy配列をPyTorchテンソルへ変換する
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)

# 分割結果を確認する
print("train:", X_train_t.shape[0], "val:", X_val_t.shape[0], "test:", X_test_t.shape[0])

train: 90 val: 15 test: 45


In [41]:
## MLPの定義

model = nn.Sequential(
    nn.Linear(4, 5),
    nn.ReLU(),
    nn.Linear(5, 3),
    #nn.Softmax(dim=1),
)

## 損失関数の定義
criterion = nn.CrossEntropyLoss()

## optimizer（パラメータ更新アルゴリズム）の定義
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [44]:
# 最大更新回数（パラメータ更新ステップ数）を500にする
max_epochs = 500

# 学習ループを回す
for epoch in range(1, max_epochs + 1):
    # 勾配を初期化する
    optimizer.zero_grad()

    # 訓練データで順伝播し，損失を計算する
    train_logits = model(X_train_t)
    train_loss = criterion(train_logits, y_train_t)

    # 逆伝播で勾配を計算し，パラメータを更新する
    train_loss.backward()
    optimizer.step()

    # 定期的に訓練損失とバリデーション精度を表示する
    if epoch % 20 == 0 or epoch == 1:
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_pred = val_logits.argmax(dim=1)
            val_acc = (val_pred == y_val_t).float().mean().item()
        print(f"epoch={epoch:3d}  train_loss={train_loss.item():.4f}  val_acc={val_acc:.4f}")

# 最後にテスト精度を評価する
with torch.no_grad():
    test_logits = model(X_test_t)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test_t).float().mean().item()

print(f"test_acc={test_acc:.4f}")

epoch=  1  train_loss=0.3308  val_acc=0.8000
epoch= 20  train_loss=0.3263  val_acc=0.8000
epoch= 40  train_loss=0.3217  val_acc=0.8000
epoch= 60  train_loss=0.3173  val_acc=0.8000
epoch= 80  train_loss=0.3130  val_acc=0.8000
epoch=100  train_loss=0.3088  val_acc=0.8000
epoch=120  train_loss=0.3048  val_acc=0.8000
epoch=140  train_loss=0.3009  val_acc=0.8000
epoch=160  train_loss=0.2970  val_acc=0.8000
epoch=180  train_loss=0.2933  val_acc=0.8667
epoch=200  train_loss=0.2897  val_acc=0.8667
epoch=220  train_loss=0.2861  val_acc=0.8667
epoch=240  train_loss=0.2827  val_acc=0.8667
epoch=260  train_loss=0.2793  val_acc=0.8667
epoch=280  train_loss=0.2760  val_acc=0.8667
epoch=300  train_loss=0.2728  val_acc=0.8667
epoch=320  train_loss=0.2696  val_acc=0.8667
epoch=340  train_loss=0.2666  val_acc=0.8667
epoch=360  train_loss=0.2636  val_acc=0.8667
epoch=380  train_loss=0.2606  val_acc=0.8667
epoch=400  train_loss=0.2577  val_acc=0.8667
epoch=420  train_loss=0.2549  val_acc=0.8667
epoch=440 